<a href="https://colab.research.google.com/github/KonradGonrad/PyTorch-deep-learning/blob/main/QMNIST.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import torchvision
from torchvision import datasets
import torch
from torchvision import transforms
import pandas
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
from pathlib import Path
import requests
import os
import random
import numpy as np
import pandas as pd

In [ ]:
# Paths
data_path = Path('data/')
train_path = data_path / 'train'
test_path = data_path / 'test'

In [ ]:
# Transformation
transformation = torchvision.transforms.Compose([
    transforms.ToTensor(),
    transforms.Resize(size=(28, 28))
])

In [ ]:
# Dataset
train_dataset = datasets.QMNIST(root=train_path, train=True, download=True, transform = transformation)
test_dataset = datasets.QMNIST(root=test_path, train=True, download=True, transform = transformation)

In [ ]:
# DataLoader
batch_size = 16
num_workers = os.cpu_count()


train_dataloader = torch.utils.data.DataLoader(dataset = train_dataset,
                                               batch_size = batch_size,
                                               shuffle = True,
                                               num_workers = num_workers)

test_dataloader = torch.utils.data.DataLoader(dataset = test_dataset,
                                              batch_size = batch_size,
                                              shuffle = False,
                                              num_workers = num_workers)

In [ ]:
# Class names
class_names = train_dataset.classes

In [ ]:
batch_images, batch_labels = next(iter(train_dataloader))

random_idx = random.randint(0, len(batch_images))
random_img, random_label = batch_images[random_idx], batch_labels[random_idx]
plt.imshow(random_img.squeeze(0), cmap='gray')
plt.axis('off')
plt.title(class_names[random_label])
plt.show()

In [ ]:
def plot_images(num_images: int = 3,
                num_columns: int = 3,
                dataloader: torch.utils.data.DataLoader = train_dataloader,
                ):
  """
  Function that shows n images on the plot with n columns.
  Images are taken from dataloader and showed by matplotlib
  """
  add_rows = (num_images - 1) // num_columns
  batch_images, batch_labels = next(iter(dataloader))

  fig, ax = plt.subplots(nrows=1 + add_rows, ncols=num_columns, figsize=(10,7))

  ax = ax.flatten() if isinstance(ax, np.ndarray) else [ax]

  random_idx = random.sample(range(len(batch_images)), k=num_images)
  random_images = batch_images[random_idx]
  random_labels = batch_labels[random_idx]

  for i in range(num_images):
    ax[i].imshow(random_images[i].squeeze(0), cmap='gray')
    ax[i].axis('off')
    ax[i].set_title(f'{class_names[random_labels[i]]}')

  for j in range(num_images, ax.size):
    fig.delaxes(ax[j])

  plt.tight_layout()
  plt.show();

In [ ]:
plot_images(num_images = 7,
            num_columns = 2,
            dataloader = train_dataloader)

In [ ]:
def train_step(model: torch.nn,
               dataloader: torch.utils.data.dataloader,
               optimizer: torch.optim,
               loss_fn: torch.nn,
               device: torch.device):
  model.to(device)
  model.train()

  train_acc, train_loss = 0, 0

  for batch, (X, y) in enumerate(dataloader):
    X, y = X.to(device), y.to(device)

    y_logits = model(X)

    loss = loss_fn(y_logits, y)
    train_loss += loss.item()

    y_pred = torch.argmax(torch.softmax(y_logits, dim=1), dim=1)
    train_acc += (y_pred == y).sum().item()/len(y_pred)

    optimizer.zero_grad()

    loss.backward()

    optimizer.step()

  train_acc /= len(dataloader)
  train_loss /= len(dataloader)
  return train_loss, train_acc

In [ ]:
def test_step(model: torch.nn,
              dataloader: torch.utils.data.dataloader,
              loss_fn: torch.nn,
              device: torch.device):
  model.to(device)
  model.eval()

  test_acc, test_loss = 0, 0

  with torch.inference_mode():
    for batch, (X, y) in enumerate(dataloader):
      X, y = X.to(device), y.to(device)

      y_logits = model(X)

      loss = loss_fn(y_logits, y)
      test_loss += loss.item()

      y_pred = torch.argmax(torch.softmax(y_logits, dim=1), dim=1)
      test_acc += ((y_pred == y).sum().item() / len(y_pred))

  test_loss /= len(dataloader)
  test_acc /= len(dataloader)
  return test_loss, test_acc

In [ ]:
import inspect

def retrieve_name(var):
    callers_local_vars = inspect.currentframe().f_back.f_locals.items()
    return [var_name for var_name, var_val in callers_local_vars if var_val is var]


In [ ]:
def train(model: torch.nn,
          train_dataloader: torch.utils.data.dataloader,
          test_dataloader: torch.utils.data.dataloader,
          optimizer: torch.optim,
          loss_fn: torch.nn,
          epochs: int,
          device: torch.device,
          model_name: str = None):

  results = {'epochs':[],
             'train_loss':[],
             'train_accuracy':[],
             'test_loss':[],
             'test_accuracy':[],
             }

  for epoch in range(epochs):
    train_loss, train_acc = train_step(model = model,
                                       dataloader = train_dataloader,
                                       optimizer = optimizer,
                                       loss_fn = loss_fn,
                                       device = device)
    test_loss, test_acc = test_step(model = model,
                                    dataloader = test_dataloader,
                                    loss_fn = loss_fn,
                                    device = device)
    print(f'Epoch: {epoch} | train_loss: {train_loss:.2f} | train_accuracy: {train_acc:.2f} | test_loss: {test_loss:.2f} | test_accuracy: {test_acc:.2f}')

    results['epochs'].append(epoch)
    results['train_loss'].append(train_loss)
    results['train_accuracy'].append(train_acc)
    results['test_loss'].append(test_loss)
    results['test_accuracy'].append(test_acc)

  return results

In [ ]:
from torch import nn
class TinyVGG(nn.Module):
  def __init__(self,
               input_shape: int,
               hidden_units: int,
               output_shape: int) -> None:
    super().__init__()
    self.conv_block_1 = nn.Sequential(
        nn.Conv2d(in_channels=input_shape,
                  out_channels=hidden_units,
                  kernel_size=3,
                  stride=1,
                  padding=0),
        nn.ReLU(),
        nn.Conv2d(in_channels=hidden_units,
                  out_channels=hidden_units,
                  kernel_size=3,
                  stride=1,
                  padding=0),
        nn.ReLU(),
        nn.MaxPool2d(kernel_size=2,
                     stride=2)
    )
    self.conv_block_2 = nn.Sequential(
        nn.Conv2d(in_channels=hidden_units,
                  out_channels=hidden_units,
                  kernel_size=3,
                  stride=1,
                  padding=0),
        nn.ReLU(),
        nn.Conv2d(in_channels=hidden_units,
                  out_channels=hidden_units,
                  kernel_size=3,
                  stride=1,
                  padding=0),
        nn.ReLU(),
        nn.MaxPool2d(kernel_size=2,
                     stride=2)
    )
    self.classifier_layer = nn.Sequential(
        nn.Flatten(),
        nn.Linear(in_features=hidden_units*4*4,
                  out_features=output_shape)
    )
  def forward(self, x):
    #print(x.shape)
    x = self.conv_block_1(x)
    #print(x.shape)
    x = self.conv_block_2(x)
    #print(x.shape)
    x = self.classifier_layer(x)
    #print(x.shape)
    return x

In [ ]:
torch.manual_seed(42)
torch.cuda.manual_seed(42)

EPOCHS = 5
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
model_0 = TinyVGG(input_shape = 1,
                  hidden_units=10,
                  output_shape=len(train_dataset.classes)).to(DEVICE)

loss_fn = torch.nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(params=model_0.parameters(),
                             lr = 0.001)

model_0_results = train(model=model_0,
                        train_dataloader=train_dataloader,
                        test_dataloader=test_dataloader,
                        optimizer=optimizer,
                        loss_fn=loss_fn,
                        device=DEVICE,
                        epochs=EPOCHS)

In [ ]:
def model_pred_and_plot(model: torch.nn,
                        test_dataloader: torch.utils.data.DataLoader):
  image_batch, label_batch = next(iter(test_dataloader))
  random_idx = random.randint(0, len(image_batch))

  image = image_batch[random_idx].unsqueeze(0)
  label = label_batch[random_idx]

  model_0.to(DEVICE)

  with torch.inference_mode():
    image, label = image.to(DEVICE), label.to(DEVICE)
    y_pred = torch.argmax(torch.softmax(model_0(image), dim=1), dim=1)
    plt.imshow(image.cpu().squeeze(), cmap='gray')
    plt.axis('off')
    title = f'Predict: {class_names[y_pred]} | Correct: {class_names[label]}'
    plt.title(title, color='g' if y_pred.item() == label.item() else 'r')

model_pred_and_plot(model=model_0,
                    test_dataloader=test_dataloader)

In [ ]:
model_0_results

In [ ]:
from typing import Literal

def plot_results(results: dict,
              name: str,
              res_type: Literal['loss', 'accuracy']):
  df = pd.DataFrame(results)
  plt.plot(df['epochs'], df[f'train_{res_type}'], label=f'Train_{res_type}', color='blue')
  plt.plot(df['epochs'], df[f'test_{res_type}'], label=f'Test_{res_type}',
          color='purple')
  plt.title(f'Model loss - {name}')
  plt.xlabel('Epochs')
  plt.ylabel('Loss')
  plt.xticks(df['epochs'])
  plt.legend()
  plt.show()

plot_results(model_0_results, retrieve_name(model_0)[0], 'accuracy')

In [ ]:
from typing import Literal

def fig_results(results: dict,
                name: str,
                res_type: Literal['loss', 'accuracy']):
  df = pd.DataFrame(results)
  fig, ax = plt.subplots()
  ax.plot(df['epochs'], df[f'train_{res_type}'], label=f'Train_{res_type}', color='blue')
  ax.plot(df['epochs'], df[f'test_{res_type}'], label=f'Test_{res_type}', color='purple')
  ax.set_title(f'{res_type} - {name}')
  ax.set_xlabel('Epochs')
  ax.set_ylabel(f'{res_type.capitalize()}')
  ax.set_xticks(df['epochs'])
  ax.legend()

  return fig, ax

In [ ]:
from matplotlib.ticker import PercentFormatter

def plot_results(results: dict,
                 name: str):
  df = pd.DataFrame(results)
  fig, ax = plt.subplots(nrows= 1, ncols = 2, figsize =(15,7))
  fig.suptitle(f'{name} results:')

  ax[0].plot(df['epochs'], df['train_loss'], label = 'Train_loss', color='blue')
  ax[0].plot(df['epochs'], df['test_loss'], label = 'Test_loss', color='purple')
  ax[0].set_xticks(df['epochs'])
  ax[0].set_ylabel('Loss')
  ax[0].set_xlabel('Epochs')
  ax[0].legend()

  ax[1].plot(df['epochs'], df['train_accuracy'], label = 'train_accuracy', color='blue')
  ax[1].plot(df['epochs'], df['test_accuracy'], label = 'test_accuracy', color = 'purple')
  ax[1].set_xticks(df['epochs'])
  ax[1].set_xlabel('Epochs')
  ax[1].set_ylabel('Accuracy')
  ax[1].yaxis.set_major_formatter(PercentFormatter(xmax=1))
  ax[1].legend()
plot_results(results = model_0_results,
             name = retrieve_name(model_0)[0])

In [ ]:
# This model is based on LeNet5 architecture,
# I used here wiki site as my research paper to recreate this model
# Link: https://en.wikipedia.org/wiki/LeNet
class LeNet5_1(nn.Module):
  """
  LeNet5 architecture for images of size (32 x 32)
  """
  def __init__(self,
               input_shape: int,
               output_shape: int) -> None:
    super().__init__()
    self.c1_layer = nn.Sequential(
        nn.Conv2d(in_channels = input_shape,
                  out_channels = 6,
                  kernel_size = 5,
                  stride=1,
                  padding=0),
        nn.ReLU()
    )
    self.s2_layer = nn.Sequential(
        nn.MaxPool2d(kernel_size = 2,
                     stride= 2,
                     padding=0)
    )
    self.c3_layer = nn.Sequential(
        nn.Conv2d(in_channels = 6,
                  out_channels = 16,
                  kernel_size = 5,
                  stride = 1,
                  padding = 0),
        nn.ReLU()
    )
    self.s4_layer = nn.Sequential(
        nn.MaxPool2d(kernel_size = 2,
                     stride = 2,
                     padding = 0)
    )
    self.c5_layer = nn.Sequential(
        nn.Conv2d(in_channels = 16,
                  out_channels = 120,
                  kernel_size = 5,
                  stride = 1,
                  padding = 0),
        nn.ReLU()
    )
    self.f1_layer = nn.Sequential(
        nn.Flatten(),
        nn.Linear(in_features = 120,
                  out_features = 84)
    )
    self.f2_layer = nn.Sequential(
        nn.ReLU(),
        nn.Linear(in_features = 84,
                  out_features = output_shape)
    )

  def forward(self, x):
    #print(f'Input shape: {x.shape}')
    x = self.c1_layer(x)
    #print(f'conv_1 shape: {x.shape}')
    x = self.s2_layer(x)
    #print(f'maxP_1 shape: {x.shape}')
    x = self.c3_layer(x)
    #print(f'conv_2 shape: {x.shape}')
    x = self.s4_layer(x)
    #print(f'maxP_2 shape: {x.shape}')
    x = self.c5_layer(x)
    #print(f'conv_3 shape: {x.shape}')
    x = self.f1_layer(x)
    #print(f'f1 shape: {x.shape}')
    x = self.f2_layer(x)
    #print(f'f2 shape: {x.shape}')
    return x


class LeNet5_2(nn.Module):
  """
  LeNet5 architecture for images of size (28 x 28)
  """
  def __init__(self,
               input_shape: int = 1,
               output_shape: int = 10
               ):
    super().__init__()
    self.layer_1 = nn.Sequential(
        nn.Conv2d(in_channels = input_shape,
                  out_channels = 6,
                  kernel_size = 5,
                  stride = 1,
                  padding = 0),
        nn.ReLU(),
        nn.AvgPool2d(kernel_size = 2,
                     stride = 2,
                     padding = 0)
    )
    self.layer_2 = nn.Sequential(
        nn.Conv2d(in_channels = 6,
                  out_channels = 16,
                  kernel_size = 5,
                  stride = 1,
                  padding = 0),
        nn.ReLU(),
        nn.AvgPool2d(kernel_size = 2,
                     stride = 2,
                     padding = 0)
    )
    self.layer_3 = nn.Sequential(
        nn.Conv2d(in_channels = 16,
                  out_channels = 120,
                  kernel_size = 4,
                  stride = 1,
                  padding = 0)
    )
    self.layer_4 = nn.Sequential(
        nn.Flatten(),
        nn.Linear(in_features = 120,
                  out_features=84)
    )
    self.layer_5 = nn.Sequential(
        nn.ReLU(),
        nn.Linear(in_features = 84,
                  out_features = output_shape)
    )
  def forward(self, x):
    x = self.layer_1(x)
    #print(x.shape)
    x = self.layer_2(x)
    #print(x.shape)
    x = self.layer_3(x)
    #print(x.shape)
    x = self.layer_4(x)
    #print(x.shape)
    x = self.layer_5(x)
    #print(x.shape)
    return x

In [ ]:
size = 32
input_shape = 1
output_shape = len(class_names)

In [ ]:
dummy_1 = torch.rand(1, size, size)
dummy_2 = torch.rand(1, size-4, size-4)

In [ ]:
model_1 = LeNet5_1(input_shape = input_shape,
                   output_shape = output_shape)

results = model_1(dummy_1.unsqueeze(0))
print(results)
print(torch.argmax(results))

In [ ]:
model_1 = LeNet5_2(input_shape=input_shape,
                   output_shape = output_shape)

results = model_1(dummy_2.unsqueeze(0))
print(results)
print(torch.argmax(results))

In [ ]:
SEED = 42
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'

torch.manual_seed(SEED)
torch.cuda.manual_seed(SEED)


model_1 = LeNet5_2(input_shape = input_shape,
                   output_shape = output_shape).to(DEVICE)

optimizer = torch.optim.Adam(params = model_1.parameters(),
                             lr = 0.001)
loss_fn = torch.nn.CrossEntropyLoss()

model_1_results = train(model = model_1,
                train_dataloader = train_dataloader,
                test_dataloader = test_dataloader,
                optimizer = optimizer,
                loss_fn = loss_fn,
                epochs = EPOCHS,
                device = DEVICE)

In [ ]:
plot_results(results = model_1_results,
             name = retrieve_name(model_1)[0])